<a href="https://colab.research.google.com/github/tmzt/TrainingExperiments/blob/main/Highbay/Local/HighbaySchemaProseFinetune2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/tmzt/TrainingExperiments/blob/main/Highbay/Local/HighbaySchemaProseFinetune2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Highbay schema/prose fine-tune - v2

Fine-tunes a small instruct model to turn a plain-language automation request
into the `GeniusAst` JSON the Highbay Genius view consumes, and measures how
often it gets the whole AST exactly right.

**Runtime: `Runtime > Change runtime type > T4 GPU`.**

## The data is already clean, and that is why this notebook is short

`Highbay/Local/data/` in this repo is self-contained - raw sources, the
generator, and the cleaned outputs - so the notebook just clones and reads.
No Drive staging, no upload, no BOM handling, no merge, no normalization: all
of that happens in `clean_genius_corpus.py` and is reproducible from
`source/`.

What that script already did, so this notebook does not have to:

* **Merged both corpora** - `mobile_data_prompts.jsonl` (92) and
  `prompts.jsonl` (50) are shape-identical, so 142 records, no duplicate
  `user_input`s, 142/142 well-formed.
* **Two on-disk formats.** `mobile_data_prompts.jsonl` carried a UTF-8 BOM, so
  plain `utf-8` broke the *first* record and only the first; `prompts.jsonl`
  was a pretty-printed JSON **array** despite the `.jsonl` name, and reading it
  line-by-line does not error - it yields `schema_mutations` fragments that
  look like records. Output is true JSONL, UTF-8, no BOM.
* **`UPDATE_RECORD.payload` was spelled two ways.** It is a field->value map in
  23/23 records, but the generator wrote an object 8 times and a *stringified*
  object 15 times. Same meaning, two spellings, nothing to learn - at ~114
  training records that is teaching a coin flip. The maps won.

## What differs from v1

* **The target is the whole AST, not `ui_prompt`.** v1 put
  `output_ast["ui_prompt"]` in the assistant turn, training the model to answer
  *"Shall I create a Water Logs table...?"*. But `ui_prompt` is a FIELD OF the
  AST, so targeting the AST yields the prose for free **and** the structure the
  view runs on. `TARGET = "ui_prompt"` restores v1's behaviour.
* **Never feed the raw AST to `load_dataset("json", ...)`.** That asks Arrow to
  infer a column type per AST field, and the AST is union-typed. It refuses the
  merged corpus outright (`cannot mix struct and non-struct, non-null values`);
  on one file alone it silently unifies the two mutation shapes into a 5-field
  struct and backfills nulls, leaving **46/92** ASTs intact. Two sides, an
  input string and an output string - the output's shape is payload, not
  schema. Arrow only ever sees `{"text": str}` here.
* **A held-out split and a real metric**, with the eval run **before** training.
  v1 ran `max_steps = 60` with no eval set at all. A 3B instruct model already
  emits plausible JSON, so the post-training number alone says nothing.
* **A system message**, and `get_chat_template` without v1's `mapping=`
  argument (that maps ShareGPT-shaped data; these are role/content messages, so
  it applied to nothing).
* **Loss on the answer only**, via `train_on_responses_only`.

Kept from v1: the `torchao` pin, unsloth's `FastLanguageModel` and
`use_gradient_checkpointing = "unsloth"`, `fp16 = not is_bf16_supported()`
(correct - a T4 is Turing and has no bf16), and `save_pretrained_gguf`.

## Read this before judging the numbers

142 records is ~114 after the split. Small for teaching a JSON schema from
scratch: expect the model to lean on what the base already knows about JSON,
and read a large train/eval gap as a corpus problem rather than a
hyperparameter one. **Scope:** the corpus has `Projects` and `Expenses` but
**no `Budget` table** across 100+ tables.

In [1]:
# 1. Runtime check - which dtype does this card actually support?
import subprocess, torch

print("GPU:", subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."

BF16 = torch.cuda.is_bf16_supported()
print(f"compute capability {torch.cuda.get_device_capability()}   bf16={BF16}")
if not BF16:
    print("T4 path: fp16. Turing has no bfloat16 - any recipe hardcoding bf16=True fails here.")

GPU: Tesla T4, 15360 MiB
compute capability (7, 5)   bf16=True


In [2]:
# 2. Dependencies (kept from v1)
!pip install torchao==0.18.0
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 61.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-uxc84qnd/unsloth_832711caa03749bb83a00b517f1353fd
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-uxc84qnd/unsloth_832711caa03749bb83a00b517f1353fd
  Resolved https://github.com/unslothai/unsloth.git to commit 3746465ec7d68587f91fac29c81be6fd6d037440
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.8.3-py3-none-any.whl size=41252658 sha256=501a8b93e228c55071a85905b926c044116666f9ecd24a6ee7e4113bcb9006e0
  Stored in directory: /tmp/pip-ephem-wheel-cache-yvhreykm/wheels/60/3e/1f/e576c0705

## 3. Data - clone and read

The repo is public, so this needs no auth and no Drive. `genius_corpus_clean.jsonl`
is merged, deduplicated, normalized, true JSONL, UTF-8 without BOM.

In [3]:
# 3. Clone this repo and read the cleaned corpus.
import os, json, random, collections

REPO_URL  = "https://github.com/tmzt/TrainingExperiments.git"
CHECKOUT  = "/content/TrainingExperiments"
DATA_DIR  = f"{CHECKOUT}/Highbay/Local/data"

if not os.path.isdir(DATA_DIR):
    !git clone --depth 1 $REPO_URL $CHECKOUT

CORPUS = f"{DATA_DIR}/genius_corpus_clean.jsonl"
records = [json.loads(line) for line in open(CORPUS, encoding="utf-8") if line.strip()]

print(f"{len(records)} records from {CORPUS}")
print(dict(collections.Counter(r["output_ast"]["intent_type"] for r in records)))
print(json.dumps(records[0], indent=2)[:420], "...")

Cloning into '/content/TrainingExperiments'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 15 (delta 1), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 56.98 KiB | 4.07 MiB/s, done.
Resolving deltas: 100% (1/1), done.
142 records from /content/TrainingExperiments/Highbay/Local/data/genius_corpus_clean.jsonl
{'SCHEMA_SUGGESTION': 71, 'PIPELINE': 71}
{
  "user_input": "I need to log how much water I drink every day on my fitness app.",
  "output_ast": {
    "intent_type": "SCHEMA_SUGGESTION",
    "schema_mutations": [
      {
        "action": "CREATE_TABLE",
        "name": "Water_Logs"
      },
      {
        "action": "ADD_COLUMN",
        "table": "Water_Logs",
        "column_name": "amount_oz",
        "type": "NUMBER"
      },
      {
        "action": "A ...


In [4]:
# 4. Contract. The eval needs it, and it is a cheap guard against a bad
#    regeneration upstream - vocabularies read off the corpus, not invented.
INTENT_TYPES  = {"PIPELINE", "SCHEMA_SUGGESTION"}
TRIGGER_TYPES = {"ON_CREATE", "ON_UPDATE", "ON_DELETE", "SCHEDULED"}
ACTION_TYPES  = {"SEND_NOTIFICATION", "UPDATE_RECORD", "CALCULATE"}
COLUMN_TYPES  = {"NUMBER", "DATE", "STRING", "BOOLEAN", "RELATION"}
# TWO mutation shapes. Reading m["table"] skips every table creation, quietly.
MUTATION_KEYS = {
    "CREATE_TABLE": {"action", "name"},
    "ADD_COLUMN":   {"action", "table", "column_name", "type"},
}

def canonical(ast):
    """ONE string form per AST, so the training target and the eval comparison
    are the same definition rather than two that drift."""
    return json.dumps(ast, sort_keys=True, separators=(",", ":"))

def validate(ast):
    problems = []
    if not isinstance(ast, dict):
        return ["not an object"]
    intent = ast.get("intent_type")
    if intent not in INTENT_TYPES:
        problems.append(f"intent_type={intent!r}")
    if intent == "PIPELINE":
        p = ast.get("pipeline_ast")
        if not isinstance(p, dict):
            problems.append("PIPELINE without pipeline_ast")
        else:
            trig, act = p.get("trigger"), p.get("action")
            if not isinstance(trig, dict): problems.append("trigger missing")
            elif trig.get("type") not in TRIGGER_TYPES:
                problems.append(f"trigger.type={trig.get('type')!r}")
            if not isinstance(act, dict): problems.append("action missing")
            elif act.get("type") not in ACTION_TYPES:
                problems.append(f"action.type={act.get('type')!r}")
    if intent == "SCHEMA_SUGGESTION":
        muts = ast.get("schema_mutations")
        if not isinstance(muts, list) or not muts:
            problems.append("SCHEMA_SUGGESTION without schema_mutations")
        else:
            for i, m in enumerate(muts):
                want = MUTATION_KEYS.get(m.get("action"))
                if want is None:
                    problems.append(f"mutation[{i}].action={m.get('action')!r}")
                elif set(m) != want:
                    problems.append(f"mutation[{i}] keys {sorted(set(m))} != {sorted(want)}")
                elif m["action"] == "ADD_COLUMN" and m.get("type") not in COLUMN_TYPES:
                    problems.append(f"mutation[{i}].type={m.get('type')!r}")
    return problems

bad = [(i, p) for i, p in ((i, validate(r["output_ast"])) for i, r in enumerate(records)) if p]
print(f"{len(records) - len(bad)}/{len(records)} well-formed")
for i, p in bad[:8]:
    print("  record", i, p)
assert not bad, "the committed corpus does not validate - regenerate it"

142/142 well-formed


In [5]:
# 5. Model: 4-bit Llama-3.2-3B-Instruct + LoRA (kept from v1)
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# No mapping= : that argument maps ShareGPT-shaped data, and these are
# role/content messages built here, so in v1 it applied to nothing.
tokenizer = get_chat_template(tokenizer, chat_template = "chatml")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.3: Fast Llama patching. Transformers: 5.13.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.8.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_docz5ecf/tokenizer_config.json.


In [6]:
# 6. ChatML, and a stratified split with a fixed seed.
TARGET = "ast"          # "ast" (default) | "ui_prompt" (v1's behaviour)

SYSTEM = (
    "You convert a user's plain-language automation request into a strict JSON "
    "AST. Reply with JSON only - no prose, no code fences."
)
SYSTEM_PROSE = (
    "You restate a user's plain-language automation request as a short "
    "confirming question. Reply with one sentence."
)

def system_for():
    return SYSTEM_PROSE if TARGET == "ui_prompt" else SYSTEM

def answer_for(record):
    if TARGET == "ui_prompt":
        return record["output_ast"].get("ui_prompt", "")
    return canonical(record["output_ast"])

def to_chatml(record):
    return {"messages": [
        {"role": "system",    "content": system_for()},
        {"role": "user",      "content": record["user_input"]},
        {"role": "assistant", "content": answer_for(record)},
    ]}

if TARGET == "ast":
    recovered = [json.loads(to_chatml(r)["messages"][2]["content"]) for r in records]
    assert all(canonical(a) == canonical(r["output_ast"])
               for a, r in zip(recovered, records)), "an AST changed through ChatML"
    print(f"all {len(records)} ASTs recovered byte-identical from the assistant turn")

SEED, EVAL_FRACTION = 3407, 0.2
by_intent = collections.defaultdict(list)
for r in records:
    by_intent[r["output_ast"].get("intent_type")].append(r)

train_recs, eval_recs = [], []
rng = random.Random(SEED)
for intent, group in sorted(by_intent.items()):
    group = group[:]; rng.shuffle(group)
    cut = max(1, round(len(group) * EVAL_FRACTION))
    eval_recs += group[:cut]; train_recs += group[cut:]
rng.shuffle(train_recs); rng.shuffle(eval_recs)
print(f"train {len(train_recs)}  eval {len(eval_recs)}")

from datasets import Dataset
def to_text(record):
    return tokenizer.apply_chat_template(
        to_chatml(record)["messages"], tokenize = False, add_generation_prompt = False)

# Every column is a string here, which is the whole point - Arrow is safe.
train_ds = Dataset.from_list([{"text": to_text(r)} for r in train_recs])
print(train_ds[0]["text"][:400], "...")

all 142 ASTs recovered byte-identical from the assistant turn
train 114  eval 28
<|im_start|>system
You convert a user's plain-language automation request into a strict JSON AST. Reply with JSON only - no prose, no code fences.<|im_end|>
<|im_start|>user
For my appliances, I need to know when the warranty expires so I can get them fixed in time.<|im_end|>
<|im_start|>assistant
{"intent_type":"SCHEMA_SUGGESTION","pipeline_ast":null,"schema_mutations":[{"action":"ADD_COLUMN","co ...


## 7. Baseline, before any training

Without this number the post-training one cannot be read.

In [ ]:
# 7. Eval: exact match on the canonical AST
def prompt_for(user_input):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system_for()},
         {"role": "user",   "content": user_input}],
        tokenize = False, add_generation_prompt = True)

@torch.no_grad()
def predict(user_input, max_new_tokens = 512):
    ids = tokenizer(prompt_for(user_input), return_tensors = "pt",
                    add_special_tokens = False).to(model.device)
    out = model.generate(**ids, max_new_tokens = max_new_tokens, do_sample = False,
                         pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:],
                            skip_special_tokens = True).strip()

def parse_ast(text):
    """Models like to wrap JSON in fences or trail prose."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        text = text[4:] if text.lower().startswith("json") else text
    start = text.find("{")
    if start < 0: return None
    depth = 0
    for i, ch in enumerate(text[start:], start):
        depth += (ch == "{") - (ch == "}")
        if depth == 0:
            try: return json.loads(text[start:i + 1])
            except json.JSONDecodeError: return None
    return None

def evaluate(dataset, label):
    if TARGET != "ast":
        print(f"--- {label}: TARGET={TARGET!r}, exact-AST does not apply ---")
        for r in dataset[:3]:
            print("  in :", r["user_input"][:70])
            print("  out:", predict(r["user_input"], 128)[:120])
        return 0.0
    n = len(dataset); parsed = exact = valid = intent_ok = 0
    misses = []
    for r in dataset:
        got, want = parse_ast(predict(r["user_input"])), r["output_ast"]
        if got is None:
            misses.append((r["user_input"], "unparseable")); continue
        parsed += 1
        if not validate(got): valid += 1
        if canonical(got) == canonical(want): exact += 1
        else: misses.append((r["user_input"], canonical(got)[:140]))
        intent_ok += got.get("intent_type") == want.get("intent_type")
    print(f"--- {label}  (n={n}) ---")
    print(f"  parseable JSON  {parsed}/{n}")
    print(f"  schema-valid    {valid}/{n}")
    print(f"  EXACT AST       {exact}/{n}   <- the number that matters")
    print(f"  intent_type     {intent_ok}/{n}")
    for inp, got in misses[:5]:
        print(f"    miss {inp[:55]!r}\n         -> {got}")
    return exact / n if n else 0.0

FastLanguageModel.for_inference(model)
baseline = evaluate(eval_recs, "BASELINE (no fine-tuning)")

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

In [ ]:
# 8. Train
from trl import SFTTrainer
from transformers import TrainingArguments

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    train_dataset = train_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # v1 used max_steps=60 regardless of corpus size; epochs scale with it.
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not BF16,          # T4
        bf16 = BF16,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = SEED,
        output_dir = "outputs",
        report_to = [],
    ),
)

# Loss on the ANSWER only. Otherwise most of the gradient goes on reproducing
# the question, which at this corpus size is most of the signal.
try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<|im_start|>user\n",
        response_part    = "<|im_start|>assistant\n",
    )
    print("training on responses only")
except Exception as e:
    print("train_on_responses_only unavailable, training on the full sequence:", e)

trainer_stats = trainer.train()

In [ ]:
# 9. Eval after training, against the baseline
FastLanguageModel.for_inference(model)
tuned = evaluate(eval_recs, "AFTER FINE-TUNING")
if TARGET == "ast":
    print(f"\nexact-AST  baseline {baseline:.1%}  ->  tuned {tuned:.1%}"
          f"   (delta {tuned - baseline:+.1%})")
    _ = evaluate(train_recs[:10], "TRAIN SUBSET (memorization check)")
    print("\nNear-perfect on train while eval lags is memorization, which is the "
          f"expected shape at {len(train_recs)} records. The fix is more corpus, "
          "not more epochs.")

## 10. Where the weights go - not into this repo

`Highbay/Local/data/` is self-contained for **inputs and code**; model weights
are a different kind of thing and do not belong beside them.

* A **q4_k_m GGUF of a 3B model is roughly 2 GB**, and GitHub rejects any file
  over **100 MB** outright. Git LFS accepts large files but its free tier is
  1 GB of storage, so it does not rescue a 2 GB artifact either.
* The **LoRA adapter** is far smaller - tens of MB - so it would squeeze under
  the limit. It is still a build output, and committing one per run bloats the
  history permanently with binaries nobody diffs.

So: the **Hugging Face Hub** is the right home, and unsloth pushes straight to
it. GitHub **Releases** is the alternative if you want the artifact attached to
this repo specifically - it allows up to 2 GB per file and keeps it out of git
history. Drive still works and is what v1 did.

What SHOULD come back into the repo after a run is the **numbers** - baseline
vs tuned exact-AST - which are small, diffable, and the thing you actually want
to compare across runs.

In [1]:
# 10. Save. Drive is optional now - the data no longer comes from there.
SAVE_DIR = "/content/genius_lora_v2"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("adapter ->", SAVE_DIR)

EXPORT_GGUF     = True
PUSH_TO_HF      = False     # set a repo id below and add an HF token first
HF_REPO         = "tmzt/genius-schema-prose-v2"
SAVE_TO_DRIVE   = true

if EXPORT_GGUF:
    # ~2 GB for 3B q4_k_m. Stays on the runtime unless pushed somewhere.
    model.save_pretrained_gguf(f"{SAVE_DIR}_q4_k_m", tokenizer,
                               quantization_method = "q4_k_m")
    print("gguf -> done")
    !du -sh {SAVE_DIR}_q4_k_m

# if PUSH_TO_HF:
#     # The right home for weights: no size ceiling to fight, and the adapter and
#     # the GGUF live together under one id.
#     from huggingface_hub import notebook_login
#     notebook_login()
#     model.push_to_hub_gguf(HF_REPO, tokenizer, quantization_method = "q4_k_m")

if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/Training Data/genius_lora_v2"
    shutil.copytree(SAVE_DIR, dest, dirs_exist_ok=True)
    print("copied to", dest)

NameError: name 'model' is not defined